In [3]:

# 1. Mount Google Drive to the mandatory default location
from google.colab import drive
drive.mount('/content/drive')

import re
import pandas as pd
import numpy as np
import mne
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 2. Fixed cloud folder variables pointing inside your My Drive
folder = Path("/content/drive/MyDrive/dataset")
output_folder = Path("/content/drive/MyDrive/output-test")

eeg_channels = ["TP9", "AF7", "AF8", "TP10"]
bands = {
    "Delta": (0.5, 4),
    "Theta": (4, 8),
    "Alpha": (8, 13),
    "Beta": (13, 30),
    "Gamma": (30, 45)
}
sfreq = 256
chunk_size = 256
LABEL_MODE = "state"

def process_eeg_file(input_file):
    df = pd.read_csv(input_file)
    df.columns = df.columns.str.strip()

    for channel in eeg_channels:
        if channel not in df.columns:
            return None # Clarified explicit return for tracking execution

    data = df[eeg_channels].to_numpy().T
    info = mne.create_info(ch_names=eeg_channels, sfreq=sfreq, ch_types="eeg")
    raw = mne.io.RawArray(data, info, verbose=False)

    processed_columns = {}
    for band_name, (f_min, f_max) in bands.items():
        raw_band = raw.copy()
        raw_band.filter(l_freq=f_min, h_freq=f_max, fir_design="firwin", verbose=False)
        band_data = raw_band.get_data()

        number_of_chunks = band_data.shape[1] // chunk_size
        usable_samples = number_of_chunks * chunk_size
        band_data = band_data[:, :usable_samples]

        band_chunks = band_data.reshape(len(eeg_channels), number_of_chunks, chunk_size)
        chunk_intensities = np.mean(band_chunks ** 2, axis=2)

        for channel_index, channel_name in enumerate(eeg_channels):
            column_name = f"{channel_name}_{band_name}"
            processed_columns[column_name] = chunk_intensities[channel_index]

    processed_df = pd.DataFrame(processed_columns)
    processed_df.insert(0, "time_seconds", np.arange(len(processed_df)))

    output_folder.mkdir(exist_ok=True, parents=True) # Enabled multi-level folder creation safety
    output_file = output_folder / f"{input_file.stem}-processed.csv"
    processed_df.to_csv(output_file, index=False)

    print("Processed:", input_file.name)
    print("Saved as:", output_file.name)
    print("Shape:", processed_df.shape)
    return processed_df

def process_all_eeg_files():
    raw_files = list(folder.glob("subject[a-d]-*.csv"))
    print(f"Found {len(raw_files)} CSV files matching the pattern in {folder}")
    if not raw_files:
        print("^ that's 0 files - verify your original_data folder path in Drive.")
        return

    for input_file in raw_files:
        process_eeg_file(input_file)

# Run processing pipeline
process_all_eeg_files()

FEATURE_COLUMNS = [f"{ch}_{band}" for ch in eeg_channels for band in bands.keys()]

def label_from_filename(filename):
    if LABEL_MODE == "state":
        match = re.search(r"subject[a-d]-(relaxed|neutral|concentrating)", filename, re.IGNORECASE)
    else:
        match = re.search(r"subject([a-d])", filename, re.IGNORECASE)

    if match:
        return match.group(1).lower()

    fallback_label = re.sub(r"-processed$", "", Path(filename).stem, flags=re.IGNORECASE)
    return fallback_label.lower()

feature_frames = []
groups = []
labels = []

processed_csvs = sorted(output_folder.glob("*-processed.csv"))
print(f"\nFound {len(processed_csvs)} processed CSVs in {output_folder}")

for file in processed_csvs:
    df = pd.read_csv(file)
    missing = [c for c in FEATURE_COLUMNS if c not in df.columns]
    if missing:
        print(f"Skipping {file.name}, missing columns: {missing}")
        continue

    label = label_from_filename(file.name)
    feature_frames.append(df[FEATURE_COLUMNS])
    groups.extend([file.stem] * len(df))
    labels.extend([label] * len(df))

if not feature_frames:
    raise RuntimeError(
        "No usable features collected. Make sure process_all_eeg_files() produced outputs."
    )

X = pd.concat(feature_frames, ignore_index=True).to_numpy()
y = np.array(labels)
groups = np.array(groups)

print("\nTotal samples:", X.shape[0])
print("Classifying by:", LABEL_MODE)
print("Classes:", sorted(set(y)))
print(pd.Series(y).value_counts())

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"\nTraining samples: {X_train.shape[0]}")

baseline_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
baseline_model.fit(X_train, y_train)
baseline_train_acc = accuracy_score(y_train, baseline_model.predict(X_train))
baseline_test_acc = accuracy_score(y_test, baseline_model.predict(X_test))

model = RandomForestClassifier(n_estimators=10, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_pred)

print(f"\nTraining accuracy: {train_acc:.3f}")
print(f"Test accuracy: {test_acc:.3f}")
print(f"\n{'Model':<38}{'Train acc':>12}{'Test acc':>12}")
print(f"{'Baseline (300 trees, unlimited depth)':<38}{baseline_train_acc:>12.3f}{baseline_test_acc:>12.3f}")
print(f"{'Small (10 trees, max_depth=10)':<38}{train_acc:>12.3f}{test_acc:>12.3f}")

print("\nClassification report (small forest):")
print(classification_report(y_test, y_pred))

print("Confusion matrix (small forest):")
print(confusion_matrix(y_test, y_pred, labels=sorted(set(y))))

importances = pd.Series(model.feature_importances_, index=FEATURE_COLUMNS).sort_values(ascending=False)
print("\nTop 10 features (small forest):")
print(importances.head(10))


MessageError: Error: credential propagation was unsuccessful